In [1]:
import os
import json
import time
from pathlib import Path
from typing import Dict, Any, Tuple, Optional

import requests
import pandas as pd

In [2]:
TOKEN = os.getenv("ENSEMBLEDATA_TOKEN", "vKLIay8miZKARJKU")
ROOT = "https://ensembledata.com/apis"

In [3]:
# -----------------------
# API calls (SCRAPE ONLY)
# -----------------------
def get_user_detailed_info(username: str) -> Dict[str, Any]:
    endpoint = "/instagram/user/detailed-info"
    params = {"username": username, "token": TOKEN}
    r = requests.get(ROOT + endpoint, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [2]:
# -----------------------
# Save / load raw payloads (JSON per account)
# -----------------------
def save_json(payload: Dict[str, Any], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def safe_username(name: str) -> str:
    # keep filenames clean
    return "".join(ch for ch in name.strip() if ch.isalnum() or ch in ("_", "-", "."))

In [3]:

# -----------------------
# Parsers (PARSE ONLY)
# -----------------------

def parse_user_detailed_payload(payload: Dict[str, Any]) -> pd.DataFrame:
    d = payload.get("data", {}) or {}

    followers = (d.get("edge_followed_by") or {}).get("count", None)
    following = (d.get("edge_follow") or {}).get("count", None)
    posts_count = (d.get("edge_owner_to_timeline_media") or {}).get("count", None)

    bio_links = d.get("bio_links") or []
    bio_links_count = len(bio_links) if isinstance(bio_links, list) else None
    
    addr = d.get("business_address_json")
    if isinstance(addr, str):
        s = addr.strip()
    
        if s not in ("", "{}", "null", "None"):
            # remove outer quotes if present
            if s[:1] == '"' and s[-1:] == '"':
                s = s[1:-1]
            # fix doubled quotes from CSV escaping
            s = s.replace('""', '"')
    
            try:
                addr = json.loads(s)
            except Exception:
                addr = {}
        else:
            addr = {}
    elif not isinstance(addr, dict):
        addr = {}
    row = {
        "username": d.get("username", ""),
        "user_id": d.get("id", ""),
        "fbid": d.get("fbid", None),
        "fb_profile_biolink": d.get("fb_profile_biolink", None),
        "category_name": d.get("category_name", None),
        "is_verified": d.get("is_verified", None),
        "is_business_account": d.get("is_business_account", None),
        "is_professional_account": d.get("is_professional_account", None),
        "biography": d.get("biography", None),
        "external_url": d.get("external_url", None),
        "followers": followers,
        "following": following,
        "posts_count": posts_count,
        "bio_links_count": bio_links_count,
        "profile_pic_url_hd": d.get("profile_pic_url_hd", None),
        "is_private_detailed": d.get("is_private", None),
        #"is_verified_detailed": d.get("is_verified", None),
        "business_phone_number": d.get("business_phone_number", None),
        "business_email": d.get("business_email", None),
        "city_name": addr.get("city_name"),
        "city_id": addr.get("city_id"),
        "latitude": addr.get("latitude"),
        "longitude": addr.get("longitude"),
        "street_address": addr.get("street_address"),
        "zip_code": addr.get("zip_code"),

    }
    return pd.DataFrame([row])

In [4]:

# -----------------------
# CSV input
# -----------------------
def read_accounts_csv(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    needed = {"brand", "username"}
    missing = needed - set(df.columns.str.lower())
    # handle case where user wrote Brands/Username etc.
    if missing:
        # normalize column names
        df.columns = [c.strip().lower() for c in df.columns]
        missing2 = needed - set(df.columns)
        if missing2:
            raise ValueError(f"CSV must contain columns: {sorted(list(needed))}. Missing: {sorted(list(missing2))}")

    # normalize and drop blanks
    df["brand"] = df["brand"].astype(str).str.strip()
    df["username"] = df["username"].astype(str).str.strip()
    df = df[(df["username"] != "") & (df["username"].str.lower() != "nan")].copy()
    return df

In [5]:

# -----------------------
# Step 1: SCRAPE + SAVE RAW JSONS
# -----------------------
def scrape_and_save_raw(
    accounts_csv: str,
    out_root: str = "Instagram",
    sleep_s: float = 0.3,
) -> Tuple[Path, Path]:
    out_root = Path(out_root)
    user_detailed_dir = out_root / "user_info_detailed"

    df_accounts = read_accounts_csv(accounts_csv)

    for _, row in df_accounts.iterrows():
        username = row["username"]
        uname = safe_username(username)


        # --- user/detailed-info ---
        try:
            detailed_payload = get_user_detailed_info(username)
            save_json(detailed_payload, user_detailed_dir / f"{uname}.json")
        except Exception as e:
            save_json({"username": username, "error": str(e)}, user_detailed_dir / f"{uname}.error.json")

        time.sleep(sleep_s)

    return  user_detailed_dir

In [8]:
ig_accounts=pd.read_csv("ig_accounts.csv")
ig_accounts.head(10)

,brand,username
0,Walmart,walmart
1,NVIDIA,nvidia
2,Budweiser,budweiser
3,Coca-Cola,cocacola
4,McDonald’s,mcdonalds
5,Loreal,lorealparis
6,Moutai,moutai.china


In [9]:
if __name__ == "__main__":
    accounts_csv = "ig_accounts.csv"   # CSV with columns: brands, username
    out_root = "Instagram"
    sleep_s = 0.3

    scrape_and_save_raw(
        accounts_csv=accounts_csv,
        out_root=out_root,
        sleep_s=sleep_s
    )

    print(f"[DONE] Scraping finished. Raw JSON saved under: {out_root}/user_info and {out_root}/user_info_detailed")

[DONE] Scraping finished. Raw JSON saved under: Instagram/user_info and Instagram/user_info_detailed


In [6]:

# -----------------------
# Step 2: PARSE saved JSONS -> tables (+ merged)
# -----------------------
def parse_saved_jsons_to_tables(
    accounts_csv: str,
    in_root: str = "Instagram",
    out_root: str = "Instagram",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    in_root = Path(in_root)
    out_root = Path(out_root)
    user_detailed_dir = in_root / "user_info_detailed"

    df_accounts = read_accounts_csv(accounts_csv)

    info_rows = []
    detailed_rows = []

    for _, row in df_accounts.iterrows():
        username = row["username"]
        brand = row["brand"]
        uname = safe_username(username)

        # load raw jsons (skip if missing)
        detailed_path = user_detailed_dir / f"{uname}.json"


        if detailed_path.exists():
            payload = load_json(detailed_path)
            df2 = parse_user_detailed_payload(payload)
            df2.insert(0, "brand", brand)
            detailed_rows.append(df2)
        else:
            detailed_rows.append(pd.DataFrame([{"brand": brand, "username": username}]))

    user_detailed_df = pd.concat(detailed_rows, ignore_index=True) if detailed_rows else pd.DataFrame()


    # Save parsed tables
    out_root.mkdir(parents=True, exist_ok=True)
    user_detailed_df.to_csv(out_root / "ig_user_info.csv", index=False)

    return  user_detailed_df

In [7]:

# -----------------------
# Example CLI usage
# -----------------------
if __name__ == "__main__":
    # INPUT CSV with columns: brands, username
    accounts_csv = "ig_accounts.csv"

    # Step 2: parse saved jsons into tables + merged
    user_detailed_df = parse_saved_jsons_to_tables(
        accounts_csv=accounts_csv,
        in_root="Instagram",
        out_root="Instagram",
    )


    print("\nUSER_DETAILED_INFO (parsed)")
    #print(user_detailed_df.head(10).to_string(index=False))

    


USER_DETAILED_INFO (parsed)


/var/folders/z0/01jh3sgn3wq18wjljcjkl1x40000gn/T/ipykernel_65775/1470447007.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  user_detailed_df = pd.concat(detailed_rows, ignore_index=True) if detailed_rows else pd.DataFrame()


In [8]:
user_detailed_df

,brand,username,user_id,fbid,fb_profile_biolink,category_name,is_verified,is_business_account,is_professional_account,biography,...,profile_pic_url_hd,is_private_detailed,business_phone_number,business_email,city_name,city_id,latitude,longitude,street_address,zip_code
0,Allianz,allianz,214036577,17841400617313344,None,None,True,True,True,Insuring the world for 135 years. #StepIntoLife,...,https://instagram.fhmo2-4.fna.fbcdn.net/v/t51....,False,None,None,None,NaN,NaN,NaN,None,None
1,Amazon Web Services,amazonwebservices,1437392412,17841400492382459,None,None,True,True,True,Amazon Web Services (AWS),...,https://scontent-ham3-1.cdninstagram.com/v/t51...,False,None,None,None,NaN,NaN,NaN,None,None
2,Aramco,aramco,1815950256,17841400427708374,None,None,True,True,True,where energy is opportunity,...,https://scontent-mia5-1.cdninstagram.com/v/t51...,False,None,None,None,NaN,NaN,NaN,None,None
3,Budweiser,budweiser,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Coca-Cola,cocacola,249655166,17841401654983900,None,None,True,True,True,Classic since always.,...,https://instagram.fktp2-1.fna.fbcdn.net/v/t51....,False,None,None,None,NaN,NaN,NaN,None,None
5,Deloitte,deloitte,2866331,17841401509930044,None,None,True,True,True,"At Deloitte, we make an impact that matters.",...,https://instagram.fruh4-6.fna.fbcdn.net/v/t51....,False,None,None,None,NaN,NaN,NaN,None,None
6,Deutsche Telekom,deutschetelekom,2160231852,17841402395560028,None,None,True,True,True,"Content wie unser Netz: schnell, stabil, Magen...",...,https://scontent-fco2-1.cdninstagram.com/v/t51...,False,None,None,None,NaN,NaN,NaN,None,None
7,Google,google,1067259270,17841401778116675,None,Internet company,True,True,True,Here to help,...,https://instagram.fksc2-1.fna.fbcdn.net/v/t51....,False,None,None,None,NaN,NaN,NaN,None,None
8,Google,googlepixel,8482988871,17841408546140373,None,Community Organization,True,True,True,A perfect 10 📱,...,https://instagram.fhyd11-2.fna.fbcdn.net/v/t51...,False,None,None,"Mountain View, California",1.082126e+14,37.389670,-122.081610,None,None
9,Google,madebygoogle,4512854723,17841404521714106,None,None,True,True,True,"Designed for your everyday.\n#Pixel10, 10 Pro,...",...,https://scontent-nrt1-2.cdninstagram.com/v/t51...,False,None,None,None,NaN,NaN,NaN,None,None
